In [1]:
from __future__ import annotations

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
%load_ext tensorboard

In [13]:
import gc
import sys
import logging
from pathlib import Path

In [5]:
import mlflow

In [6]:
import torch
import numpy as np

import run


In [7]:
_logger = logging.getLogger("adaptive_milling_training")
_logger.propagate = False
_logger.setLevel(logging.DEBUG)
for _handler in _logger.handlers:
    _logger.removeHandler(_handler)

_logging_formatter = logging.Formatter(
    "%(asctime)s - %(name)s - %(levelname)s - %(message)s", datefmt="%H:%M:%S'"
)
_stream_handler = logging.StreamHandler()
_stream_handler.setLevel(logging.WARNING)
_stream_handler.setFormatter(_logging_formatter)

_logger.addHandler(_stream_handler)


In [8]:
from utils import MONAI_LOG_DIR

mlflow.pytorch.autolog()
port = 54598
mlflow_uri = f"file://{MONAI_LOG_DIR}"
_logger.info("Setting up mlflow with URI '%s'", mlflow_uri)
mlflow.set_tracking_uri(mlflow_uri)
mlflow.set_experiment("MONAI adaptive milling")

_file_handler = logging.FileHandler(MONAI_LOG_DIR / "training.log")
_file_handler.setLevel(logging.INFO)
_file_handler.setFormatter(_logging_formatter)
_logger.addHandler(_file_handler)

print(
    f"Run the following command to start:\n$mlflow ui --backend-store-uri {mlflow_uri} --port {port}\nThen navigate to:\nhttp://127.0.0.1:{port}"
)


2025/03/19 20:27:43 WARNING mlflow.utils.autologging_utils: MLflow pytorch autologging is known to be compatible with 1.9.0 <= torch <= 2.5.1, but the installed version is 2.5.1+cu118. If you encounter errors during autologging, try upgrading / downgrading torch to a compatible version, or try upgrading MLflow.


Run the following command to start:
$mlflow ui --backend-store-uri file:///home/tpr78264/logs/mlflow --port 54598
Then navigate to:
http://127.0.0.1:54598


In [9]:
%%script false --no-raise-error
from utils import TENSORBOARD_LOG_DIR

logging.info("Setting up Tensorboard with log dir %s", TENSORBOARD_LOG_DIR)
%tensorboard --logdir $TENSORBOARD_LOG_DIR

In [10]:
test_csv = "/ceph/groups/structbio/adaptive_milling_project/2024labels_new/test.csv"
all_files_csv = (
    "/ceph/groups/structbio/adaptive_milling_project/2024labels_new/all_files.csv"
)


In [ ]:
gc.collect()
torch.cuda.empty_cache()
try:
    dist_matrix = np.asarray(
        [
            [0.0, 0.1, 0.1, 0.1, 0.1, 0.1],  # padding
            [0.1, 0.0, 0.5, 0.8, 0.6, 0.2],  # background
            [0.1, 0.5, 0.0, 0.8, 0.9, 1.0],  # lamella
            [0.1, 0.8, 0.8, 0.0, 0.7, 0.7],  # GIS
            [0.1, 0.6, 0.9, 0.7, 0.0, 0.9],  # crack
            [0.1, 0.2, 1.0, 0.7, 0.9, 0.0],  # void
        ],
        dtype=np.float32,
    )
    max_epochs = 120
    frozen_fraction = 0.2
    models_dir = Path.cwd().parent / "models"
    run.run_training(
        # model_save_path = (
        #     Path.cwd().parent
        #     / "models"
        #     / "250317_234235_all_files_smp_efficientnet_b4_unetplusplus.pth"
        # )
        # mlflow_run_id = "f6453abe173949bf8564c868fa1a0528"
        # run.submit_validation_for_mlflow_run(
        #     mlflow_run_id,
        #     model_save_path,
        #     116,
        all_files_csv,
        model_name="smp_efficientnet_b4_unet",  # "segresnet",
        loss_name="diceloss",
        learning_rate=1e-3,
        epochs=max_epochs,
        image_size=768,
        frozen_epochs=int(max_epochs * frozen_fraction),
        # model_kwargs={"encoder_weights": "advprop", "pretrained": True},
        loss_kwargs={
            "weights": (0.0, 1.0, 4.0, 3.0, 6.0, 2.0),
            "dist_matrix": dist_matrix,
        },
        gpu_number=3,
    )
finally:  # noqa: E722
    with torch.no_grad():
        torch.cuda.empty_cache()
    try:
        mlflow.end_run()
    except:  # noqa: E722
        _logger.error("Failed to end MLFlow run", exc_info=True)

Training progress:   1%|          | 1/120 [00:00<?, ?epoch/s]

Epoch 1 training:   0%|          | 0/75 [00:00<?, ?step/s]

Epoch 2 training:   0%|          | 0/75 [00:00<?, ?step/s]

Epoch 2 validation:   0%|          | 0/38 [00:00<?, ?step/s]

Epoch 3 training:   0%|          | 0/75 [00:00<?, ?step/s]

...

Epoch 120 training:   0%|          | 0/75 [00:00<?, ?step/s]

Epoch 120 validation:   0%|          | 0/38 [00:00<?, ?step/s]

train completed, best metric 'mean_of_key_metrics': 0.7021 at epoch 116


You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.


Submitting validation images from best validation epoch to MLFlow:   0%|          | 0/38 [00:00<?, ?step/s]